# Telegram war-news → structured events (LLM extraction)

Runs on **Kaggle (GPU T4 x2, recommended)** or **Google Colab (T4)** for free. No paid GPU needed.

Pipeline:
1. Start a local LLM server (Ollama) and pull **Gemma 4** (best open model on the Ukrainian LLM leaderboard).
2. Load the exported Telegram posts (`*.jsonl` from `scripts/tg_export.py`).
3. **CPU step, no LLM:** parse the Air Force (`kpszsu`) morning summaries with regex → `kpszsu_daily.csv`.
4. Keyword pre-filter → only war-related posts go to the LLM.
5. **GPU step:** LLM extracts a strict JSON record per post (JSON-schema-constrained output). Results are checkpointed, so a crashed/timed-out run resumes where it stopped.
6. Save `events_llm_shard{K}.jsonl` → download → upload to Databricks bronze.

Split work between teammates with `SHARD_ID` / `NUM_SHARDS` (each runs a different shard on their own Kaggle account).

In [ ]:
# ===================== CONFIG =====================
MODEL        = "auto"   # "auto" | "gemma4:26b" (Kaggle 2xT4) | "gemma4:12b" (Colab T4) | "gemma4:e4b" (fast/smoke test)
SHARD_ID     = 0        # 0..NUM_SHARDS-1 — each teammate runs a different one
NUM_SHARDS   = 1
LIMIT        = None     # e.g. 200 for a quick test run; None = everything in this shard
NUM_PARALLEL = 8        # concurrent requests to the LLM server
MAX_CHARS    = 2500     # truncate very long posts
CHANNELS_FOR_LLM = ["lachentyt", "ssternenko", "kpszsu"]  # kpszsu: only its summary posts go to the LLM
# ==================================================
import os, glob, sys
IN_KAGGLE = os.path.exists("/kaggle")
IN_COLAB  = "google.colab" in sys.modules
OUT_DIR   = "/kaggle/working" if IN_KAGGLE else "/content/out"
os.makedirs(OUT_DIR, exist_ok=True)
print("Kaggle:", IN_KAGGLE, "| Colab:", IN_COLAB, "| out:", OUT_DIR)

## 1. Input data
- **Kaggle:** *Add Input → Upload → New Dataset*, upload the `data/telegram/*.jsonl` files. They appear under `/kaggle/input/<dataset-name>/`.
- **Colab:** run the cell; it asks you to upload the `.jsonl` files (or put them in Google Drive and change `INPUT_GLOB`).

In [ ]:
if IN_KAGGLE:
    INPUT_GLOB = "/kaggle/input/**/*.jsonl"
else:
    INPUT_GLOB = "/content/data/*.jsonl"
    if not glob.glob(INPUT_GLOB):
        from google.colab import files
        os.makedirs("/content/data", exist_ok=True)
        up = files.upload()
        for name, data in up.items():
            open(f"/content/data/{name}", "wb").write(data)
paths = sorted(glob.glob(INPUT_GLOB, recursive=True))
print(paths)
assert paths, "No .jsonl input found"

In [ ]:
import json, pandas as pd
rows = []
for p in paths:
    with open(p, encoding="utf-8") as f:
        rows += [json.loads(l) for l in f if l.strip()]
posts = (pd.DataFrame(rows)
           .drop_duplicates(["channel", "post_id"])
           .assign(datetime=lambda d: pd.to_datetime(d["datetime"], utc=True))
           .sort_values(["channel", "post_id"]).reset_index(drop=True))
posts["text"] = posts["text"].fillna("")
print(posts.groupby("channel").agg(n=("post_id", "size"), first=("datetime", "min"), last=("datetime", "max")))

## 2. Air Force summaries → numbers with regex (CPU, no LLM)
Morning posts like *"У ніч на 19 листопада … 98 ударними БпЛА … збито/подавлено 69 … влучання на 18 локаціях"*.

In [ ]:
import re
MONTHS = {"січня":1,"лютого":2,"березня":3,"квітня":4,"травня":5,"червня":6,"липня":7,
          "серпня":8,"вересня":9,"жовтня":10,"листопада":11,"грудня":12}
def _int(m): return int(m.group(1)) if m else None
def parse_af_summary(text, dt):
    if not re.search(r"(збито|подавлено)", text, re.I) or not re.search(r"(противник|ворог)\w*\s+атакував", text, re.I):
        return None
    m = re.search(r"[Уу] ніч на (\d{1,2}) (\w+)", text)
    night = None
    if m and m.group(2) in MONTHS:
        y = dt.year - (1 if (MONTHS[m.group(2)] == 12 and dt.month == 1) else 0)
        night = f"{y:04d}-{MONTHS[m.group(2)]:02d}-{int(m.group(1)):02d}"
    total_launched = _int(re.search(r"атакував\w*\s+(\d+)\s+засоб", text))
    drones = sum(int(x) for x in re.findall(r"(\d+)\s+ударн\w*\s+БпЛА", text))
    missiles = sum(int(x) for x in re.findall(r"(\d+)\s+(?:\w+\s+){0,3}ракет", text.split("збито")[0]))
    return {
        "post_id": None, "night_of": night,
        "launched_total": total_launched,
        "launched_drones": drones or None,
        "launched_missiles": missiles or None,
        "shot_down_total": _int(re.search(r"ЗБИТО/ПОДАВЛЕНО\s+(\d+)", text)) or _int(re.search(r"збито/подавлено\s+(\d+)", text)),
        "hit_locations": _int(re.search(r"на (\d+) локаці", text)),
        "launch_origins": ", ".join(re.findall(r"(Орел|Курськ|Брянськ|Міллерово|Приморсько-Ахтарськ|Шаталове|Навля|Донецьк|Крим\w*|Гвардійське|Чауда|Ростов\w*|Бєлгород\w*|Воронеж\w*)", text)) or None,
    }

af = posts[posts.channel == "kpszsu"]
recs = []
for _, r in af.iterrows():
    x = parse_af_summary(r.text, r.datetime)
    if x:
        x.update(post_id=r.post_id, posted_at=r.datetime.isoformat(), url=r.url)
        recs.append(x)
af_daily = pd.DataFrame(recs)
af_daily.to_csv(f"{OUT_DIR}/kpszsu_daily.csv", index=False)
print(len(af_daily), "summary posts parsed")
af_daily.tail(10)

## 3. Keyword pre-filter → what goes to the LLM

In [ ]:
KW = re.compile(r"(атак|удар|обстріл|ракет|шахед|shahed|бпла|дрон|балістик|кинджал|калібр|іскандер|каб\b|загин|загибл|поранен|вбит|жертв|"
                r"влучан|вибух|приліт|нпз|нафтопереробн|аеродром|енергетик|підстанц|знеструм|блекаут|"
                r"переговор|перемир|трамп|trump|путін|зеленськ|санкці|саміт|ультиматум|візит|"
                r"знищен|уражен|збито|ппо|тривог)", re.I)
sel = posts[posts.channel.isin(CHANNELS_FOR_LLM) & posts.text.str.len().gt(30)].copy()
# kpszsu: only summary posts (live-tracking one-liners are skipped)
sel = sel[(sel.channel != "kpszsu") | sel.post_id.isin(af_daily.get("post_id", pd.Series(dtype=int)))]
sel = sel[sel.text.str.contains(KW)]
sel = sel[sel.post_id % NUM_SHARDS == SHARD_ID] if NUM_SHARDS > 1 else sel
if LIMIT: sel = sel.sample(min(LIMIT, len(sel)), random_state=42)
print(sel.channel.value_counts(), "\nTotal to LLM in this shard:", len(sel))

## 4. Start the LLM server (Ollama) and pull the model

In [ ]:
import subprocess, time, shutil, torch
ngpu = torch.cuda.device_count()
vram = sum(torch.cuda.get_device_properties(i).total_memory for i in range(ngpu)) / 1e9 if ngpu else 0
if MODEL == "auto":
    MODEL = "gemma4:26b" if vram >= 30 else ("gemma4:12b" if vram >= 14 else "gemma4:e4b")
print(f"GPUs: {ngpu}, VRAM: {vram:.0f} GB -> model {MODEL}")
assert ngpu, "No GPU! Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → Change runtime type → T4 GPU"

if not shutil.which("ollama"):
    subprocess.run("apt-get -qq install -y zstd >/dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(NUM_PARALLEL), OLLAMA_CONTEXT_LENGTH="4096",
           OLLAMA_KEEP_ALIVE="-1", OLLAMA_FLASH_ATTENTION="1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{OUT_DIR}/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
subprocess.run(["ollama", "pull", MODEL], check=True)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 5. Extraction schema + prompt

In [ ]:
OBLASTS = ["Вінницька","Волинська","Дніпропетровська","Донецька","Житомирська","Закарпатська","Запорізька",
           "Івано-Франківська","Київська","м. Київ","Кіровоградська","Луганська","Львівська","Миколаївська","Одеська",
           "Полтавська","Рівненська","Сумська","Тернопільська","Харківська","Херсонська","Хмельницька","Черкаська",
           "Чернівецька","Чернігівська","АР Крим","м. Севастополь"]
SCHEMA = {
  "type": "object",
  "properties": {
    "relevant":      {"type": "boolean"},
    "event_type":    {"type": "string", "enum": ["russian_attack_on_ukraine","ukrainian_strike_on_russia","casualties_update",
                                                  "air_defense_summary","diplomacy_politics","frontline","energy_outage","other"]},
    "event_date":    {"type": ["string","null"], "description": "YYYY-MM-DD when the event happened"},
    "ua_oblasts":    {"type": "array", "items": {"type": "string", "enum": OBLASTS}},
    "ru_regions":    {"type": "array", "items": {"type": "string"}},
    "weapons":       {"type": "array", "items": {"type": "string", "enum": ["shahed","jet_shahed","decoy_drone","ballistic_missile",
                                                  "cruise_missile","kh22_onyx","glide_bomb_kab","artillery","fpv_drone","long_range_ua_drone","other"]}},
    "launched":      {"type": ["integer","null"]},
    "shot_down":     {"type": ["integer","null"]},
    "killed":        {"type": ["integer","null"]},
    "injured":       {"type": ["integer","null"]},
    "children_killed": {"type": ["integer","null"]},
    "target_type":   {"type": "string", "enum": ["energy","residential","civil_infrastructure","military","oil_refinery_or_depot",
                                                  "airfield","industry_defense","transport","unknown"]},
    "is_count_update": {"type": "boolean", "description": "true if post revises an earlier casualty/attack count"},
    "actors":        {"type": "array", "items": {"type": "string"}, "description": "politicians/orgs mentioned, e.g. Trump, Putin, Zelensky"},
    "diplomacy_signal": {"type": "string", "enum": ["none","peace_talks_positive","peace_talks_negative","threat_or_ultimatum","sanctions","aid_to_ukraine"]},
    "summary_en":    {"type": "string", "description": "<= 20 words"}
  },
  "required": ["relevant","event_type","event_date","ua_oblasts","ru_regions","weapons","launched","shot_down","killed","injured",
               "children_killed","target_type","is_count_update","actors","diplomacy_signal","summary_en"]
}
SYSTEM = (
 "Ти аналітик даних про російсько-українську війну. Читаєш один пост з Telegram-каналу і повертаєш ЛИШЕ JSON за схемою.\n"
 "Правила: витягуй тільки факти, явно сказані в пості; якщо числа немає — null; не вигадуй. "
 "event_date — дата події (не публікації); 'вчора'/'у ніч на' рахуй від дати публікації. "
 "ua_oblasts — області України, де стались удари/жертви. ru_regions — регіони РФ, якщо українські удари по Росії. "
 "killed/injured — цивільні жертви саме цієї події. Якщо пост не про війну — relevant=false, event_type='other'."
)
def make_messages(r):
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Канал: {r.channel}\nДата публікації (UTC): {r.datetime:%Y-%m-%d %H:%M}\nПост:\n{r.text[:MAX_CHARS]}"}]

## 6. Run extraction (resumable, parallel)

In [ ]:
import requests, concurrent.futures as cf, threading
from tqdm.auto import tqdm
OUT = f"{OUT_DIR}/events_llm_shard{SHARD_ID}.jsonl"
done = set()
# resume from a previous Kaggle run: attach its Output as an Input and the file is picked up here
prev = glob.glob(f"/kaggle/input/**/events_llm_shard{SHARD_ID}.jsonl", recursive=True)
if prev and not os.path.exists(OUT):
    shutil.copy(prev[0], OUT); print("resuming from", prev[0])
if os.path.exists(OUT):
    done = {(d["channel"], d["post_id"]) for d in map(json.loads, open(OUT, encoding="utf-8"))}
todo = [r for r in sel.itertuples() if (r.channel, r.post_id) not in done]
print(f"already done: {len(done)}, to do: {len(todo)}")
lock = threading.Lock()

def extract(r):
    for attempt in range(3):
        try:
            resp = requests.post("http://127.0.0.1:11434/api/chat", timeout=300, json={
                "model": MODEL, "messages": make_messages(r), "stream": False, "format": SCHEMA,
                "think": False, "options": {"temperature": 0, "num_ctx": 4096, "num_predict": 400}})
            resp.raise_for_status()
            out = json.loads(resp.json()["message"]["content"])
            return {"channel": r.channel, "post_id": int(r.post_id), "posted_at": r.datetime.isoformat(),
                    "url": r.url, "model": MODEL, **out}
        except Exception as e:
            err = repr(e)
    return {"channel": r.channel, "post_id": int(r.post_id), "error": err, "model": MODEL}

t0 = time.time()
with open(OUT, "a", encoding="utf-8") as f, cf.ThreadPoolExecutor(NUM_PARALLEL) as ex:
    for res in tqdm(ex.map(extract, todo), total=len(todo)):
        with lock:
            f.write(json.dumps(res, ensure_ascii=False) + "\n"); f.flush()
el = time.time() - t0
print(f"{len(todo)} posts in {el/60:.1f} min ({len(todo)/max(el,1):.2f} posts/s)")

## 7. Sanity check + quick look

In [ ]:
ev = pd.read_json(OUT, lines=True)
print("errors:", ev.get("error", pd.Series(dtype=str)).notna().sum(), "/", len(ev))
ok = ev[ev.get("error", pd.Series(index=ev.index, dtype=object)).isna()] if "error" in ev else ev
print(ok.event_type.value_counts())
ok[["posted_at","channel","event_type","ua_oblasts","weapons","launched","shot_down","killed","injured","summary_en"]].head(20)

## 8. (Optional) Evaluate against hand labels → model choice
Put a CSV `labels.csv` with columns `channel, post_id, event_type, killed, injured, launched, shot_down` (≈200 hand-labelled posts) into the input dataset.
Run the notebook once per model (`gemma4:26b`, `gemma4:12b`, …) with `LIMIT=None` restricted to those posts and compare. Log the metrics into MLflow on Databricks later.

In [ ]:
lab_paths = glob.glob("/kaggle/input/**/labels.csv", recursive=True) + glob.glob("/content/data/labels.csv")
if lab_paths:
    lab = pd.read_csv(lab_paths[0])
    m = lab.merge(ok, on=["channel","post_id"], suffixes=("_true","_pred"))
    print("n =", len(m))
    print("event_type accuracy:", (m.event_type_true == m.event_type_pred).mean().round(3))
    for c in ["killed","injured","launched","shot_down"]:
        t, p = m[f"{c}_true"], m[f"{c}_pred"]
        print(f"{c:10s} exact match: {((t == p) | (t.isna() & p.isna())).mean():.3f}")
else:
    print("no labels.csv — skipping evaluation")

## 9. Download results
Kaggle: outputs in `/kaggle/working` appear in the notebook's **Output** tab after *Save Version → Save & Run All (Commit)*.
Colab: run the cell below.

In [ ]:
if IN_COLAB:
    from google.colab import files
    for p in glob.glob(f"{OUT_DIR}/*.jsonl") + glob.glob(f"{OUT_DIR}/*.csv"):
        files.download(p)
print(os.listdir(OUT_DIR))